# Simple Time Series Forecasting with StatsForecast and AutoETS

In this example, we will forecast **monthly coffee sales**.

We will learn the difference between:

- `StatsForecast` — manages the forecasting process.
- `AutoETS()` — the statistical forecasting model.
- `freq="MS"` — tells StatsForecast that our observations occur monthly.
- `season_length` — tells AutoETS how many observations make up one seasonal cycle.
- `h` — tells StatsForecast how many future observations we want to forecast.
- `plot_series()` — plots the historical data and forecasts.

In [1]:
import pandas as pd

from statsforecast import StatsForecast
from statsforecast.models import AutoETS
from utilsforecast.plotting import plot_series

## 1. Create the data

Suppose we have monthly coffee sales.

Our dataset needs three important columns:

- `unique_id` = identifies the time series.
- `ds` = the date/time column.
- `y` = the value we want to forecast.

Here, `y` represents the number of coffees sold each month.

In [2]:
data = pd.DataFrame({
    "unique_id": ["coffee"] * 24,

    "ds": pd.date_range(
        start="2024-01-01",
        periods=24,
        freq="MS"
    ),

    "y": [
        # 2024
        100, 110, 130, 150, 180, 220,
        250, 240, 190, 160, 120, 105,

        # 2025
        110, 120, 140, 160, 190, 230,
        260, 250, 200, 170, 130, 115
    ]
})

data

,unique_id,ds,y
0,coffee,2024-01-01,100
1,coffee,2024-02-01,110
2,coffee,2024-03-01,130
3,coffee,2024-04-01,150
4,coffee,2024-05-01,180
5,coffee,2024-06-01,220
6,coffee,2024-07-01,250
7,coffee,2024-08-01,240
8,coffee,2024-09-01,190
9,coffee,2024-10-01,160


## 2. Understanding `freq="MS"`

Our observations occur once every month.

Therefore, we use:

    freq="MS"

`MS` means **Month Start**.

For example:

- 2024-01-01 → January observation
- 2024-02-01 → February observation
- 2024-03-01 → March observation

So:

    freq="MS"

answers the question:

**"How often do I have an observation?"**

Answer: **Monthly**

In [3]:
data[["ds", "y"]]

,ds,y
0,2024-01-01,100
1,2024-02-01,110
2,2024-03-01,130
3,2024-04-01,150
4,2024-05-01,180
5,2024-06-01,220
6,2024-07-01,250
7,2024-08-01,240
8,2024-09-01,190
9,2024-10-01,160


## 3. Understanding `season_length=12`

Now we ask a DIFFERENT question:

**"After how many observations might the seasonal pattern repeat?"**

Our data are monthly.

Therefore:

    1 observation = 1 month

If we expect a yearly seasonal pattern:

    1 year = 12 months
           = 12 observations

Therefore:

    season_length = 12

Important:

`season_length=12` does NOT tell Python that the data are monthly.

Instead:

- `freq="MS"` → observations occur monthly.
- `season_length=12` → one seasonal cycle contains 12 observations.

Together, they mean:

**"I have monthly data and I expect a yearly seasonal pattern."**

## 4. Create the AutoETS model

`AutoETS()` is the actual statistical forecasting model.

ETS models can model components such as:

- Level
- Trend
- Seasonality

Here we use:

    AutoETS(season_length=12)

because we have monthly data and want to allow for a yearly seasonal cycle.

In [4]:
model = AutoETS(
    season_length=12
)

model

AutoETS

## 5. Create the StatsForecast object

`StatsForecast` manages the forecasting process.

We give it:

1. The forecasting model we want to use.
2. The frequency of our observations.

Here:

    models=[model]

means:

"Use our AutoETS model."

And:

    freq="MS"

means:

"My observations occur monthly."

In [5]:
sf = StatsForecast(
    models=[model],
    freq="MS"
)

## 6. Forecast future months

Now we can ask StatsForecast to produce forecasts.

`h` means **forecast horizon**.

It answers:

**"How many future observations do I want to predict?"**

Because our observations are monthly:

    h=1  → forecast 1 month
    h=3  → forecast 3 months
    h=6  → forecast 6 months
    h=12 → forecast 12 months

Here we will forecast the next 6 months.

In [6]:
forecast = sf.forecast(
    df=data,
    h=6
)

forecast

,unique_id,ds,AutoETS
0,coffee,2026-01-01,120.000214
1,coffee,2026-02-01,129.996704
2,coffee,2026-03-01,149.999374
3,coffee,2026-04-01,170.001419
4,coffee,2026-05-01,200.002121
5,coffee,2026-06-01,239.997009


## 7. Plot the historical data and forecast

`plot_series()` is used to visualize the time series.

It allows us to see:

- historical observations
- future forecasts

on a graph.

In [7]:
plot_series(
    data,
    forecast
)

<Figure size 1600x350 with 1 Axes>

# Summary

There are three different concepts that should not be confused.

## `freq="MS"`

Tells StatsForecast how often observations occur.

    freq="MS"

means:

**One observation every month.**

---

## `season_length=12`

Tells AutoETS how many observations make up one seasonal cycle.

Because our observations are monthly:

    12 observations
    = 12 months
    = 1 year

Therefore:

    season_length=12

means:

**Allow for a seasonal pattern that repeats approximately every year.**

---

## `h=6`

Tells StatsForecast how many future observations to forecast.

Because our observations are monthly:

    h=6

means:

**Forecast the next 6 months.**

---

## Putting everything together

    freq="MS"
        ↓
    My observations occur MONTHLY.

    season_length=12
        ↓
    A seasonal cycle contains 12 observations.
        ↓
    12 monthly observations = 1 year.

    h=6
        ↓
    Forecast the next 6 observations.
        ↓
    6 monthly observations = 6 months.

So:

    StatsForecast(
        models=[AutoETS(season_length=12)],
        freq="MS"
    )

can be read as:

**"Use AutoETS to forecast my monthly data, allowing for a seasonal cycle of 12 monthly observations."**